# Momentum Strategy Live Trading Dashboard

**Daily rebalance across a multi-asset crypto universe.**

| Step | Cell | Action |
|------|------|--------|
| 1 | Cell 2 | Configure universe, capital, variant |
| 2 | Cell 3 | Pre-launch sizing preview |
| 3 | Cell 4 | Launch momentum strategy subprocess |
| 4 | Cell 5 | Status overview |
| 5 | Cell 6 | Weight table + dashboard (auto-refresh) |
| 6 | Cell 7 | Stop process |

In [1]:
# STEP 1: CONFIGURATION
# ========================================================
import sys, os, pathlib, warnings
warnings.filterwarnings('ignore')

# Resolve src/ directory
_candidates = [
    pathlib.Path(globals().get('_dh', ['.'])[0]).resolve().parent / 'src',
    pathlib.Path.cwd().parent / 'src',
    pathlib.Path.cwd() / 'src',
]
SRC_DIR = None
for _p in _candidates:
    if (_p / 'AQM_Momentum_Live.py').exists():
        SRC_DIR = str(_p)
        break
if SRC_DIR is None:
    raise FileNotFoundError(
        f'Could not locate src/AQM_Momentum_Live.py. Tried: {[str(p) for p in _candidates]}'
    )
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

_IS_WINDOWS = sys.platform == 'win32'
if _IS_WINDOWS:
    VENV_PYTHON = os.path.join(SRC_DIR, '..', '.venv', 'Scripts', 'python.exe')
else:
    VENV_PYTHON = os.path.join(SRC_DIR, '..', 'PairsTrading', 'bin', 'python')

SCRIPT_PATH = os.path.join(SRC_DIR, 'AQM_Momentum_Live.py')
NOTEBOOKS_DIR = os.path.join(SRC_DIR, '..', 'notebooks')

# ── Strategy configuration ──
CAPITAL       = 1_000
VARIANT       = 'turtle'    # 'turtle' or 'tanh'
SHORT_WINDOW  = 5
LONG_WINDOW   = 30
MAX_WEIGHT    = 0.10
STOP_LOSS     = -0.10
REBALANCE_UTC = 0           # Midnight UTC
MIN_REBALANCE = 0.005       # 0.5% min weight change
EXCHANGE      = 'binance'   # 'binance' or 'bitget'
TESTNET       = True
BATCH_N       = 3
BATCH_INTERVAL = 600
LIMIT_OFFSET  = 2

TOKENS = [
    'AAVE', 'AIXBT', 'AVAX', 'BCH', 'BNB', 'BTC', 'COMP', 'DOGE', 'DOT',
    'DYDX', 'EIGEN', 'ENA', 'ETH', 'ETHFI', 'FORM', 'INJ', 'JUP', 'LTC',
    'NEAR', 'PNUT', 'RAY', 'SOL', 'SUI', 'TRX', 'UNI', 'WIF',
]

print(f'Config OK | {len(TOKENS)} tokens | ${CAPITAL:,} | {VARIANT.upper()} '
      f'| S={SHORT_WINDOW} L={LONG_WINDOW} | {EXCHANGE.upper()}')

Config OK | 26 tokens | $1,000 | TURTLE | S=5 L=30 | BINANCE


In [2]:
# STEP 2: PRE-LAUNCH SIZING PREVIEW
# ========================================================
import requests, numpy as np, pandas as pd
from IPython.display import display, HTML

def _fetch_price(symbol, exchange='binance'):
    sym = f'{symbol}USDT'
    if exchange == 'binance':
        url = 'https://fapi.binance.com/fapi/v1/ticker/price'
        r = requests.get(url, params={'symbol': sym}, timeout=10)
        return float(r.json()['price'])
    else:
        url = 'https://api.bitget.com/api/v2/mix/market/ticker'
        r = requests.get(url, params={'symbol': sym, 'productType': 'USDT-FUTURES'}, timeout=10)
        return float(r.json()['data'][0]['lastPr'])

rows = []
for token in TOKENS:
    try:
        price = _fetch_price(token, EXCHANGE)
        max_notional = MAX_WEIGHT * CAPITAL
        rows.append({'Token': token, 'Price': f'${price:.4g}',
                     f'Max Notional ({MAX_WEIGHT*100:.0f}%)': f'${max_notional:,.0f}',
                     f'Max Qty': f'{max_notional/price:,.2f}'})
    except Exception as e:
        rows.append({'Token': token, 'Error': str(e)})

display(HTML(f'''
<div style="background:#0f3460; color:white; padding:8px 16px;
            border-radius:6px; margin-bottom:8px;">
    <b>Sizing preview</b> | {EXCHANGE.upper()} | Capital ${CAPITAL:,}
    | MaxWeight {MAX_WEIGHT*100:.0f}% | {len(TOKENS)} tokens
</div>
'''))
display(pd.DataFrame(rows))

,Token,Price,Max Notional (10%),Max Qty,Error
0,AAVE,$82.84,$100,1.21,NaN
1,AIXBT,$0.01994,$100,"5,015.05",NaN
2,AVAX,$6.211,$100,16.10,NaN
3,BCH,$190.6,$100,0.52,NaN
4,BNB,$559.5,$100,0.18,NaN
5,BTC,$5.963e+04,$100,0.00,NaN
6,COMP,$15.61,$100,6.41,NaN
7,DOGE,$0.07483,$100,"1,336.36",NaN
8,DOT,$0.86,$100,116.28,NaN
9,DYDX,$0.14,$100,714.29,NaN


In [4]:
# STEP 3: LAUNCH MOMENTUM STRATEGY
# ========================================================
import subprocess, time, threading
from datetime import datetime

LOG_FILE = os.path.join(NOTEBOOKS_DIR, 'log_momentum.txt')
STATE_FILE = 'live_state_momentum.json'

# Account preset — routes orders through the momentum-specific API key
# (MOMENTUM_API_KEY / SECRET_MOMENTUM_KEY in .env)
ACCOUNT = 'binance_testnet'

cmd = [
    VENV_PYTHON, SCRIPT_PATH,
    '--account', ACCOUNT,
    '--tokens', ','.join(TOKENS),
    '--capital', str(CAPITAL),
    '--variant', VARIANT,
    '--short-window', str(SHORT_WINDOW),
    '--long-window', str(LONG_WINDOW),
    '--max-weight', str(MAX_WEIGHT),
    '--stop-loss', str(STOP_LOSS),
    '--rebalance-utc', str(REBALANCE_UTC),
    '--min-rebalance', str(MIN_REBALANCE),
    '--batch-n', str(BATCH_N),
    '--batch-interval', str(BATCH_INTERVAL),
    '--limit-offset-bps', str(LIMIT_OFFSET),
    '--state-file', STATE_FILE,
]

log_fh = open(LOG_FILE, 'w')
popen_kwargs = dict(stdout=log_fh, stderr=subprocess.STDOUT, cwd=SRC_DIR)
if _IS_WINDOWS:
    popen_kwargs['creationflags'] = subprocess.CREATE_NEW_PROCESS_GROUP
else:
    popen_kwargs['preexec_fn'] = os.setsid

proc = subprocess.Popen(cmd, **popen_kwargs)

print(f'  Momentum strategy launched  PID={proc.pid}  account={ACCOUNT}')
print(f'  State file: {STATE_FILE}')
print(f'  Log file:   {os.path.basename(LOG_FILE)}')

  Momentum strategy launched  PID=20108  account=binance_testnet
  State file: live_state_momentum.json
  Log file:   log_momentum.txt


In [5]:
# STEP 4: STATUS OVERVIEW
# ========================================================
import json, time
from IPython.display import display, HTML

alive = proc.poll() is None
status = 'Running' if alive else f'Stopped ({proc.returncode})'
json_path = os.path.join(NOTEBOOKS_DIR, STATE_FILE)
json_age = None
if os.path.exists(json_path):
    json_age = time.time() - os.path.getmtime(json_path)

print(f'PID: {proc.pid}  Status: {status}')
print(f'JSON age: {json_age:.0f}s' if json_age else 'JSON: not yet created')

PID: 20108  Status: Running
JSON age: 3166s


In [ ]:
# STEP 5: AUTO-REFRESH DASHBOARD
# ========================================================
import json, time, os
import pandas as pd
from datetime import datetime
from IPython.display import display, HTML, clear_output

REFRESH_SECONDS = 60

def load_momentum_state():
    path = os.path.join(NOTEBOOKS_DIR, STATE_FILE)
    if not os.path.exists(path):
        return None
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except Exception:
        return None

def render_momentum_dashboard(state):
    if state is None:
        display(HTML('<h3>Waiting for first snapshot...</h3>'))
        return

    ts = state.get('timestamp', '?')[:19]
    strat = state.get('strategy', {})
    acct = state.get('account', {})
    weights = state.get('weights', {})
    positions = state.get('positions', {})

    n_long = strat.get('n_long', 0)
    n_short = strat.get('n_short', 0)
    gross = strat.get('gross_exposure', 0)
    net = strat.get('net_exposure', 0)
    variant = strat.get('variant', '?').upper()

    total_pnl = 0.0
    for nemo, ps in positions.items():
        pnl = ps.get('unrealized_pnl')
        if pnl is not None:
            total_pnl += pnl

    cash = acct.get('cash') or 0
    equity = acct.get('total_equity') or 0
    alive_now = proc.poll() is None

    # Header
    pnl_color = 'green' if total_pnl >= 0 else 'red'
    display(HTML(f'''
    <div style="background:#0f3460; color:white; padding:10px 20px;
                border-radius:8px; margin-bottom:8px;">
        <h2 style="margin:0;">Momentum Dashboard
            <span style="font-size:0.5em; color:#aaa;">{ts} UTC</span></h2>
        <p style="margin:4px 0 0 0;">
            {'Running' if alive_now else 'Stopped'} |
            {variant} S={strat.get('short_window')} L={strat.get('long_window')} |
            Long={n_long} Short={n_short} |
            Gross={gross:.3f} Net={net:.4f} |
            <span style="color:{pnl_color}">uPnL ${total_pnl:+,.2f}</span> |
            Equity ${equity:,.0f}
        </p>
    </div>
    '''))

    # Weight table
    if weights:
        rows = []
        for nemo in sorted(weights.keys()):
            w = weights[nemo]
            ps = positions.get(nemo, {})
            rows.append({
                'Token': nemo,
                'Direction': w.get('direction', 'FLAT'),
                'Target W': f"{w.get('target_weight', 0):+.4f}",
                'Current W': f"{w.get('current_weight', 0):+.4f}",
                'Delta': f"{w.get('weight_delta', 0):+.4f}",
                'Pos Value': f"${w.get('position_value', 0):,.0f}",
                'uPnL': f"${ps.get('unrealized_pnl', 0) or 0:+,.2f}",
            })
        display(pd.DataFrame(rows))

try:
    while True:
        clear_output(wait=True)
        state = load_momentum_state()
        render_momentum_dashboard(state)
        display(HTML(
            f'<p style="color:gray;">Auto-refresh every {REFRESH_SECONDS}s '
            f'- interrupt kernel to stop</p>'
        ))
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print('Auto-refresh stopped.')

,Token,Direction,Target W,Current W,Delta,Pos Value,uPnL
0,AAVE,SHORT,-0.0318,+0.0000,-0.0318,$0,$+0.00
1,AIXBT,SHORT,-0.0443,-0.1429,+0.0986,$-200,$-3.10
2,AVAX,SHORT,-0.0712,+0.0000,-0.0712,$0,$+0.00
3,BCH,SHORT,-0.0801,+0.0000,-0.0801,$0,$+0.00
4,BNB,SHORT,-0.0449,+0.0000,-0.0449,$0,$+0.00
5,BTC,SHORT,-0.0661,-0.2150,+0.1489,$-300,$-3.15
6,COMP,SHORT,-0.0142,+0.0000,-0.0142,$0,$+0.00
7,DOGE,SHORT,-0.0502,+0.0000,-0.0502,$0,$+0.00
8,DOT,SHORT,-0.0560,+0.0000,-0.0560,$0,$+0.00
9,DYDX,SHORT,-0.0500,+0.0000,-0.0500,$0,$+0.00


In [ ]:
# STEP 6: STOP PROCESS
# ========================================================
import signal as _signal

if proc.poll() is None:
    try:
        if _IS_WINDOWS:
            proc.terminate()
            proc.wait(timeout=5)
        else:
            os.killpg(os.getpgid(proc.pid), _signal.SIGTERM)
            proc.wait(timeout=5)
    except Exception:
        proc.kill()
    log_fh.close()
    print(f'Momentum strategy stopped (PID={proc.pid})')
else:
    print(f'Process already stopped (exit={proc.returncode})')

In [ ]:
# VIEW LOG (last 60 lines)
# ========================================================
TAIL_LINES = 60

if os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'r') as f:
        lines = f.readlines()
    tail = lines[-TAIL_LINES:]
    print(f'Log ({len(lines)} total lines, showing last {TAIL_LINES}):\n')
    print(''.join(tail))
else:
    print('No log file yet.')